# Zoom — Logistic Regression — Breast Cancer Wisconsin (Diagnostic)

**Course:** UE 5BM753 — Supervised and Unsupervised Learning for Biology
**Part of:** the Breast Cancer Wisconsin (Diagnostic) final report, split into
per-step notebooks. See `00_Final_Report_.ipynb` for the combined narrative and
`README.md` for the overall conclusion.

**Question addressed here:** can we reliably predict the diagnosis from the selected features?

Tunes and evaluates the final Logistic Regression model, on the features selected in `04_Feature_Selection.ipynb`.

## Setup

Defines the paths used below and imports the reusable functions kept in
`01_scripts/`.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

sys.path.append(os.path.join("..", "01_scripts"))
from evaluation import evaluate_classifier
from plots import plot_roc_curves, plot_confusion_matrices

# This notebook only needs the files exported by 04_Feature_Selection.ipynb, not
# its in-memory variables, so it can run entirely on its own.
RANDOM_STATE = 42
CV_FOLDS = 10
RESULTS_DIR = os.path.join("..", "04_results")
FIGURES_DIR = os.path.join("..", "05_figures")
PROCESSED_DATA_DIR = os.path.join("..", "02_data", "processed")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

## Hyperparameter tuning

A 10-fold cross-validated grid search tunes the regularization strength `C` of the
model, over a logarithmic grid from 0.001 to 100. Recall is the score being optimized,
for the same clinical reason as before: minimizing missed malignant cases matters more
than avoiding false alarms.

In [ ]:
train_df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "04_train_selected_features.csv"))
X_train_sel = train_df.drop(columns=["label"]).values
y_train = train_df["label"].values

MODEL_NAME = "Logistic Regression (selected features)"

param_grid = {"C": np.logspace(-3, 2, 30)}
grid_logreg = GridSearchCV(
    LogisticRegression(random_state=RANDOM_STATE, max_iter=5000),
    param_grid, cv=CV_FOLDS, scoring='recall', n_jobs=-1
)
grid_logreg.fit(X_train_sel, y_train)

best_idx = grid_logreg.best_index_
cv_recall_std = grid_logreg.cv_results_["std_test_score"][best_idx]

print(f"Best C (final model): {grid_logreg.best_params_['C']:.4f}")
print(f"Mean recall on training-fold CV (model selection): "
      f"{grid_logreg.best_score_:.4f} +/- {cv_recall_std:.4f}\n")

results_cv = pd.DataFrame(grid_logreg.cv_results_)
plt.figure(figsize=(9, 5))
plt.plot(results_cv["param_C"].astype(float), results_cv["mean_test_score"],
          marker='o', color='teal', alpha=0.85)
plt.axvline(grid_logreg.best_params_['C'], linestyle='--', color='crimson', alpha=0.7,
            label=f"Optimal C = {grid_logreg.best_params_['C']:.3f}")
plt.xscale('log')
plt.title(f"Recall ({CV_FOLDS}-fold CV) vs. C (final Logistic Regression, selected features fixed)")
plt.xlabel("C (log scale)")
plt.ylabel("Mean recall (CV)")
plt.legend()
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "05_logreg_c_tuning.png"))
plt.show()

**Findings.** The grid search selects C = 30.39, near the top of the tested range:
once restricted to these 7 features, the model needs little regularization to
generalize well. The cross-validated recall at this C reaches 0.951 ± 0.055 on the
training folds, a fairly narrow spread, so this is not a lucky pick tied to one
particular split.

## Evaluation

The optimal `C` is retained, and the final model is refit on the entire restricted
training set. It is then evaluated on the held-out test set, using the same 6 metrics
as the earlier screening. ROC curves and confusion matrices are also produced, to
visualize class separation and error types directly.

In [ ]:
# Test set only, loaded from the same file 04_Feature_Selection.ipynb exported: kept
# separate from the training set so no test information leaks into the tuning above.
test_df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "04_test_selected_features.csv"))
X_test_sel = test_df.drop(columns=["label"]).values
y_test = test_df["label"].values

best_logreg = grid_logreg.best_estimator_
models_dict = {MODEL_NAME: best_logreg}

metrics_logreg = evaluate_classifier(best_logreg, X_train_sel, y_train, X_test_sel, y_test)
df_results_logreg = pd.DataFrame([metrics_logreg], index=[MODEL_NAME])
df_results_logreg.to_csv(os.path.join(RESULTS_DIR, "05_logreg_final_metrics.csv"))

print("--- Final evaluation (held-out test set) ---")
display(df_results_logreg.round(4))

plot_roc_curves(
    classifiers=models_dict, X_tr=X_train_sel, y_tr=y_train, X_te=X_test_sel, y_te=y_test,
    out_path=os.path.join(FIGURES_DIR, "05_logreg_roc_curve.png"))

plot_confusion_matrices(
    classifiers=models_dict, X_tr=X_train_sel, y_tr=y_train, X_te=X_test_sel, y_te=y_test,
    out_path=os.path.join(FIGURES_DIR, "05_logreg_confusion_matrix.png")
)

**Findings.** On the held-out test set, the tuned model misses 3 malignant cases
out of 64 (recall = 0.953), the clinically costliest type of error, while raising
only 1 false alarm among the 62 cases it flags as malignant (precision = 0.984).
ROC-AUC reaches 0.997, meaning it ranks malignant cases above benign ones correctly
for almost every possible decision threshold, not just the one used here.

## Discussion

Logistic Regression was chosen for this zoom because it had the lowest average rank
across all 6 screening metrics, with a narrow, stable recall spread across folds, and
because it stays directly interpretable: one coefficient per feature, with a clear
sign and magnitude. On the held-out test set, the tuned model misses only 3 malignant
cases out of 64.

**Limitation.** Prioritizing recall trades off some precision: more benign cases may
get flagged for unnecessary follow-up. This is accepted here given the higher clinical
cost of missing a real cancer, but it is a real limitation of the chosen scoring
priority, not a free win. The dataset also comes from a single institution, so this
performance is not guaranteed to hold on data from other hospitals or populations.